In [0]:
df = spark.table("novacart_dev.bronze.products")
display(df)

In [0]:
from pyspark.sql import functions as F
df = df.toDF(*[c.strip().lower().replace(" ", "_") for c in df.columns])

In [0]:
df = df.replace(["\\N", "?", "", "null", "NULL"], None)

In [0]:
string_cols = [c for c, t in df.dtypes if t == "string"]

for col in string_cols:
    df = df.withColumn(col, F.trim(F.lower(F.col(col))))

In [0]:
df = df.dropDuplicates(["product_id"])  

In [0]:
df = df.withColumn("price", F.col("price").cast("double"))

df = df.withColumn(
    "price",
    F.when(F.col("price") <= 0, None).otherwise(F.col("price"))
)

In [0]:
df = df.withColumn("currency", F.upper(F.col("currency")))
df = df.withColumn("country_code", F.upper(F.col("country_code")))

In [0]:
df = df.withColumn(
    "dq_missing_product_id",
    F.when(F.col("product_id").isNull(), 1).otherwise(0)
)

df = df.withColumn(
    "dq_missing_product_name",
    F.when(F.col("product_name").isNull(), 1).otherwise(0)
)

df = df.withColumn(
    "dq_invalid_price",
    F.when(F.col("price").isNull(), 1).otherwise(0)
)

In [0]:
df = df.withColumn("load_timestamp", F.current_timestamp())

In [0]:
df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("novacart_dev.silver.slv_products")

In [0]:
display(spark.table("novacart_dev.silver.slv_products"))